# Kiểm tra GPU

In [1]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
import os
import sys
import platform
from pathlib import Path
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)


Python: 3.11.9
PyTorch: 2.9.0+cpu
CUDA available: False


# Cài thư viện

In [2]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
# !pip install -q git+https://github.com/KaiyangZhou/deep-person-reid.git

In [3]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
!pip install -q ultralytics opencv-python pandas numpy matplotlib seaborn tqdm scikit-learn pyyaml scipy Pillow gdown ipywidgets
!pip install -q lap cython_bbox


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\DUNG\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\DUNG\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Xác định project root

In [4]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
from pathlib import Path
import os
import sys

# Vast.ai thường dùng /workspace
# Colab thường dùng /content
candidate_roots = [
    Path("/workspace/ticket_suspicion_project"),
    Path("/content/ticket_suspicion_project"),
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
]

PROJECT_ROOT = None

for p in candidate_roots:
    if (p / "requirements.txt").exists() or (p / "notebooks").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()

PROJECT_ROOT = PROJECT_ROOT.resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)

sys.path.append(str(PROJECT_ROOT))


PROJECT_ROOT: D:\hoiancomputervision\ticket_suspicion_project


In [5]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
from src.config import ensure_project_dirs

ensure_project_dirs(PROJECT_ROOT)

print("Project directories checked.")


Project directories checked.


# Tạo config mặc định nếu chưa có

In [6]:
# Tạo file cấu hình mặc định cho pipeline nếu người dùng chưa cung cấp các file cần thiết.
from src.config import save_yaml
from pathlib import Path

pipeline_path = PROJECT_ROOT / "configs/pipeline_config.yaml"
camera_path = PROJECT_ROOT / "configs/camera_config.yaml"
roi_path = PROJECT_ROOT / "configs/roi_config.yaml"
scoring_path = PROJECT_ROOT / "configs/scoring_config.yaml"

if not pipeline_path.exists():
    pipeline_config = {
        "project_name": "ticket_suspicion_multicamera",
        "detector": {
            "model": "yolov8n.pt",
            "person_class_id": 0,
            "confidence_threshold": 0.35,
            "iou_threshold": 0.5,
            "image_size": 960
        },
        "tracker": {
            "type": "bytetrack",
            "tracker_yaml": "bytetrack.yaml",
            "track_thresh": 0.25,
            "match_thresh": 0.8,
            "track_buffer": 30
        },
        "tracklet": {
            "min_frames": 10,
            "min_duration_sec": 1.0,
            "min_bbox_height": 40,
            "top_k_crops": 5
        },
        "reid": {
            "model": "osnet_x1_0",
            "embedding_dim": 512,
            "input_height": 256,
            "input_width": 128,
            "device": "cuda"
        },
        "association": {
            "similarity_threshold": 0.55,
            "w_reid": 0.6,
            "w_time": 0.3,
            "w_quality": 0.1
        },
        "runtime": {
            "max_frames_debug": 500,
            "fps_default": 30.0
        }
    }
    save_yaml(pipeline_config, pipeline_path)

if not camera_path.exists():
    camera_config = {
        "cameras": {
            "cam_entrance_01": {
                "source_type": "frame_sequence",
                "source_path": "data/raw/MOT17",
                "fps": None,
                "start_time_sec": 0.0,
                "sequence_keyword": "MOT17-02-FRCNN"
            },
            "cam_ticket_01": {
                "source_type": "simulation",
                "source_path": None,
                "fps": 30.0,
                "start_time_sec": 0.0
            }
        }
    }
    save_yaml(camera_config, camera_path)

if not roi_path.exists():
    roi_config = {
        "cam_entrance_01": {
            "entrance_line": {
                "points": [[100, 500], [900, 500]],
                "direction": "down"
            }
        },
        "cam_ticket_01": {
            "ticket_counter_zone": {
                "polygon": [[300, 200], [700, 200], [750, 600], [250, 600]],
                "min_dwell_time_sec": 5.0
            },
            "queue_zone": {
                "polygon": [[100, 250], [280, 250], [280, 650], [100, 650]],
                "min_dwell_time_sec": 8.0
            }
        }
    }
    save_yaml(roi_config, roi_path)

if not scoring_path.exists():
    scoring_config = {
        "scoring": {
            "alert_threshold": 0.70,
            "watch_threshold": 0.50,
            "max_travel_time_sec": 600.0,
            "current_time_sec": 1000.0
        },
        "weights": {
            "entry": 0.25,
            "no_ticket": 0.30,
            "time_exceeded": 0.20,
            "track_quality": 0.10,
            "assoc_confidence": 0.15
        }
    }
    save_yaml(scoring_config, scoring_path)

print("Config files are ready.")


Config files are ready.


# Kiểm tra dataset MOT17

In [7]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
from pathlib import Path

raw_dir = PROJECT_ROOT / "data/raw"
print("Raw data directory:", raw_dir)

mot_candidates = list(raw_dir.rglob("img1"))
print("Found img1 directories:", len(mot_candidates))

for i, p in enumerate(mot_candidates[:10]):
    print(i, p)


Raw data directory: D:\hoiancomputervision\ticket_suspicion_project\data\raw
Found img1 directories: 14
0 D:\hoiancomputervision\ticket_suspicion_project\data\raw\MOT17\test\MOT17-01-FRCNN\img1
1 D:\hoiancomputervision\ticket_suspicion_project\data\raw\MOT17\test\MOT17-03-FRCNN\img1
2 D:\hoiancomputervision\ticket_suspicion_project\data\raw\MOT17\test\MOT17-06-FRCNN\img1
3 D:\hoiancomputervision\ticket_suspicion_project\data\raw\MOT17\test\MOT17-07-FRCNN\img1
4 D:\hoiancomputervision\ticket_suspicion_project\data\raw\MOT17\test\MOT17-08-FRCNN\img1
5 D:\hoiancomputervision\ticket_suspicion_project\data\raw\MOT17\test\MOT17-12-FRCNN\img1
6 D:\hoiancomputervision\ticket_suspicion_project\data\raw\MOT17\test\MOT17-14-FRCNN\img1
7 D:\hoiancomputervision\ticket_suspicion_project\data\raw\MOT17\train\MOT17-02-FRCNN\img1
8 D:\hoiancomputervision\ticket_suspicion_project\data\raw\MOT17\train\MOT17-04-FRCNN\img1
9 D:\hoiancomputervision\ticket_suspicion_project\data\raw\MOT17\train\MOT17-05-FRCN